# Pediatric Cohort — PRR-Filtered Overview
## Demographics, Reporting & Severity Overview

**Dataset:** Pediatric — PRR-filtered signals only (PRR lower 95% CI > 1)  
**Source:** `data/notebook/output/signal_analysis/filtered_prr/`  
**Output:** `data/notebook/output/top_terms_pediatric/`

In [ ]:
# ============================================================
# Setup + Data Loading
# ============================================================
from pathlib import Path
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_style("whitegrid")

ROOT = Path("..").resolve()
SIGNAL_OUT = ROOT / "data" / "notebook" / "output" / "signal_analysis"
FULL_DATA_ROOT = ROOT / "data" / "output"

# ── Dataset configuration ──────────────────────────────────────────────
MODE = "pediatric"
PRR_FILE = SIGNAL_OUT / "filtered_prr" / "pediatric_prr_filtered.parquet"
SOURCE_PATHS = [FULL_DATA_ROOT / "Pediatric" / "patient_report_reporter_drug_reaction_full_data.parquet"]
COHORT_LABEL = "Pediatric"

OUTPUT_DIR = ROOT / "data" / "notebook" / "output" / "top_terms_pediatric"
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

# ── Load PRR-filtered pair list ────────────────────────────────────────
if not PRR_FILE.exists():
    raise FileNotFoundError(f"Run signal_filtering_and_stratification.ipynb first: {PRR_FILE}")
prr_pairs = pl.read_parquet(PRR_FILE)
print(f"PRR-filtered pairs:     {prr_pairs.height:,}")

pair_key_set = set(zip(prr_pairs["drug"].to_list(), prr_pairs["event"].to_list()))
print(f"Unique (drug, event):   {len(pair_key_set):,}")

# ── Load full event data ───────────────────────────────────────────────
print(f"\nLoading full event data for {COHORT_LABEL} ...")
dfs = [pd.read_parquet(p) for p in SOURCE_PATHS]
df_full = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
del dfs
print(f"Full event rows:        {len(df_full):,}")

# ── Subset to PRR-filtered pairs ───────────────────────────────────────
_key = df_full["medicinal_product"].astype(str) + "||" + df_full["reaction_meddrapt"].astype(str)
_pair_keys = {f"{d}||{e}" for d, e in pair_key_set}
df = df_full[_key.isin(_pair_keys)].copy()
del df_full, _key, _pair_keys
print(f"Filtered event rows:    {len(df):,}  (rows belonging to PRR-passed pairs)")

print(f"\nUnique reports:       {df['safetyreportid'].nunique():,}")
print(f"Unique drugs:           {df['medicinal_product'].nunique():,}")
print(f"Unique reactions:       {df['reaction_meddrapt'].nunique():,}")

---
# A — Demographics, Reporting & Severity

Overview of the Pediatric cohort (PRR-filtered signals only).

## A.1 — Dataset Summary Table

In [ ]:
# Dataset overview table
summary_rows = [
    {"Metric": "Filtered event rows",          "Value": f"{len(df):,}"},
    {"Metric": "Unique reports (safetyreportid)", "Value": f"{df['safetyreportid'].nunique():,}"},
    {"Metric": "Unique drugs (medicinal_product)", "Value": f"{df['medicinal_product'].nunique():,}"},
    {"Metric": "Unique reactions (PT)",        "Value": f"{df['reaction_meddrapt'].nunique():,}"},
    {"Metric": "Unique (drug, event) pairs",   "Value": f"{len(pair_key_set):,}"},
    {"Metric": "Date range (receive_date)",    "Value": f"{pd.to_datetime(df['receive_date']).min().date()} → {pd.to_datetime(df['receive_date']).max().date()}"},
    {"Metric": "Female %",                     "Value": f"{(df['patient_sex'] == 'Female').sum() / df['safetyreportid'].nunique() * 100:.1f}%" if df.height > 0 else "N/A"},
    {"Metric": "Male %",                       "Value": f"{(df['patient_sex'] == 'Male').sum() / df['safetyreportid'].nunique() * 100:.1f}%" if df.height > 0 else "N/A"},
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(OUTPUT_DIR / "tables" / "dataset_summary.csv", index=False)

## A.2 — Age Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ages = df.loc[df["age_years"].notna(), "age_years"]
if len(ages) > 0:
    ax.hist(ages, bins=50, color="#2196F3" if MODE == "adult" else "#FF9800" if MODE == "pediatric" else "#9C27B0",
            edgecolor="white", alpha=0.85)
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Count (event rows)")
    ax.set_title(f"Age Distribution — {COHORT_LABEL}  (n={len(ages):,})")
    ax.axvline(ages.median(), color="red", linestyle="--", alpha=0.7,
               label=f"Median = {ages.median():.0f} years")
    ax.legend()
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "age_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# NICHD bands (pediatric only)
if "nichd" in df.columns and df["nichd"].notna().any():
    nichd_order = ["infancy", "toddler", "early_childhood", "middle_childhood", "early_adolescence", "late_adolescence"]
    nichd_counts = df["nichd"].value_counts().reindex(nichd_order).dropna()
    if len(nichd_counts) > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.barh(nichd_counts.index.tolist()[::-1], nichd_counts.values[::-1],
                color="#FF9800", edgecolor="white")
        ax.set_xlabel("Count (event rows)")
        ax.set_title(f"NICHD Age Bands — {COHORT_LABEL}")
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "figures" / "nichd_bands.png", dpi=150, bbox_inches="tight")
        plt.show()

## A.3 — Sex Distribution

In [ ]:
sex_counts = df.groupby("safetyreportid")["patient_sex"].first().value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
colors_map = {"Female": "#E91E63", "Male": "#1976D2", "Unknown": "#9E9E9E"}
colors = [colors_map.get(s, "#757575") for s in sex_counts.index]
ax.pie(sex_counts.values, labels=sex_counts.index, autopct="%1.1f%%",
       colors=colors, startangle=90)
ax.set_title(f"Sex Distribution — {COHORT_LABEL}  ({sex_counts.sum():,} unique reports)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "sex_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

display(pd.DataFrame({"Sex": sex_counts.index, "Count": sex_counts.values,
                      "%": (sex_counts.values / sex_counts.sum() * 100).round(2)}))

## A.4 — Reports Over Time

In [ ]:
df["_year"] = pd.to_datetime(df["receive_date"]).dt.year
year_reports = df.groupby("_year")["safetyreportid"].nunique().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
color_map = {"adult": "#2196F3", "pediatric": "#FF9800", "combined": "#9C27B0"}
ax.bar(year_reports.index, year_reports.values,
       color=color_map.get(MODE, "#757575"), edgecolor="white", alpha=0.85)
ax.set_xlabel("Year")
ax.set_ylabel("Unique Reports")
ax.set_title(f"Reports by Year — {COHORT_LABEL}")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for x, y in zip(year_reports.index, year_reports.values):
    ax.text(x, y + max(year_reports.values)*0.01, f"{y:,}",
            ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "reports_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

## A.5 — Top Reporter Countries

In [ ]:
country_counts = (
    df.groupby("safetyreportid")["reporter_country"].first()
    .value_counts().head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
color_map = {"adult": "#2196F3", "pediatric": "#FF9800", "combined": "#9C27B0"}
ax.barh(country_counts.index[::-1], country_counts.values[::-1],
        color=color_map.get(MODE, "#757575"), edgecolor="white")
ax.set_xlabel("Unique Reports")
ax.set_title(f"Top 15 Reporter Countries — {COHORT_LABEL}")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "reporter_country.png", dpi=150, bbox_inches="tight")
plt.show()

## A.6 — Severity Breakdown

Report-level flags (death, hospitalization, etc.) for Pediatric.

In [ ]:
severity_flags = ["serious", "death", "hospitalization", "life_threatening", "disabling", "congenital_anomali", "other"]
available = [f for f in severity_flags if f in df.columns]

# Use safetyreportid-level (report counts), not row counts
rep_df = df.drop_duplicates(subset=["safetyreportid"])[["safetyreportid"] + available]
n_reports = len(rep_df)

sev_rows = []
for flag in available:
    cnt = (rep_df[flag] == 1).sum()
    sev_rows.append({"Outcome": flag, "Reports": int(cnt), "%": round(cnt/n_reports*100, 2) if n_reports > 0 else 0})
sev_df = pd.DataFrame(sev_rows).sort_values("Reports", ascending=False)
display(sev_df)
sev_df.to_csv(OUTPUT_DIR / "tables" / "severity_breakdown.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
color_map = {"adult": "#2196F3", "pediatric": "#FF9800", "combined": "#9C27B0"}
ax.barh(sev_df["Outcome"][::-1], sev_df["Reports"][::-1],
        color=color_map.get(MODE, "#757575"), edgecolor="white", alpha=0.85)
ax.set_xlabel("Reports (unique safetyreportid)")
ax.set_title(f"Severity Outcomes — {COHORT_LABEL}  (n={n_reports:,} unique reports)")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(ax.patches, sev_df["Reports"][::-1]):
    ax.text(bar.get_width() + max(sev_df["Reports"])*0.01, bar.get_y() + bar.get_height()/2,
            f"{val:,} ({val/n_reports*100:.1f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "severity_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary — Pediatric

Key descriptive statistics exported to:
- `top_terms_pediatric/tables/dataset_summary.csv`
- `top_terms_pediatric/tables/severity_breakdown.csv`
- `top_terms_pediatric/figures/*.png`